# Model Deployment
Packaging the scaler and trained Logistic Regression model into a Scikit-Learn Pipeline, saving to a pickle file, and demonstrating real-time inference.

In [1]:
# Step 1: Import Libraries
import os
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
import joblib


In [2]:
# Step 2: Load the Dataset
data = pd.read_csv('creditcard.csv' if os.path.exists('creditcard.csv') else 'creditcard.csv.zip')
# Separate features and labels
X = data.drop(columns='Class', axis=1)
y = data['Class']


In [3]:
# Step 3: Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=2
)
print(f'Train samples: {len(X_train):,} | Test samples: {len(X_test):,}')


Train samples: 227,845 | Test samples: 56,962


In [4]:
# Step 4: Create Pipeline and Train Model
model_pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('log_reg', LogisticRegression(solver='lbfgs', max_iter=1000, class_weight='balanced', random_state=42))
])
model_pipeline.fit(X_train, y_train)

# Save trained pipeline
joblib.dump(model_pipeline, 'fraud_detection_model.pkl')
print('Model saved successfully to fraud_detection_model.pkl!')


Model saved successfully to fraud_detection_model.pkl!


In [5]:
# Reusable Real-Time Inference Function
def predict_fraud(transaction_frame):
    prediction = model_pipeline.predict(transaction_frame)
    probability = model_pipeline.predict_proba(transaction_frame)[:, 1]
    return {
        'prediction': int(prediction[0]),
        'fraud_probability': float(round(probability[0], 4)),
        'status': 'FRAUD ALERT (BLOCK)' if prediction[0] == 1 else 'APPROVED'
    }

# Test inference on sample transaction
sample = X_test.iloc[[0]]
result = predict_fraud(sample)
print('Sample Transaction Prediction:', result)


Sample Transaction Prediction: {'prediction': 0, 'fraud_probability': 0.2237, 'status': 'APPROVED'}
